In [6]:
from Bio import SeqIO
import pandas as pd

def load_hla_sequences(fasta_path):
    """加载 fasta 文件中的序列，返回 dict: allele -> sequence"""
    seq_dict = {}
    for record in SeqIO.parse(fasta_path, "fasta"):
        allele = record.id.split()[0]  # 去掉后缀 class_II
        seq = str(record.seq).replace("\n", "")
        seq_dict[allele] = seq
    return seq_dict

def compute_heterozygosity_vector(seq1, seq2):
    """两个氨基酸序列比对生成杂合性向量（0相同，1不同）"""
    if len(seq1) != len(seq2):
        return None
    return [0 if a == b else 1 for a, b in zip(seq1, seq2)]

def generate_aa_heterozygosity_file(hla_file, fasta_file, output_file, colname1="DRB11", colname2="DRB12"):
    """从型别注释生成0/1序列文件"""
    hla_df = pd.read_csv(hla_file, sep="\t")
    seq_dict = load_hla_sequences(fasta_file)

    records = []
    for _, row in hla_df.iterrows():
        id_ = row["id"]
        allele1, allele2 = row[colname1], row[colname2]
        if allele1 not in seq_dict or allele2 not in seq_dict:
            continue
        vec = compute_heterozygosity_vector(seq_dict[allele1], seq_dict[allele2])
        if vec is None:
            continue
        records.append({
            "id": id_,
            "allele1": allele1,
            "allele2": allele2,
            "hetero_vec": ",".join(map(str, vec))
        })
    aa_anno_df = pd.DataFrame(records)
    aa_anno_df.to_csv(output_file, index=False)
    return aa_anno_df


In [8]:
aa_anno_df = generate_aa_heterozygosity_file("./data/DRB1_anno.txt", "./data/hla_exon_sequences.fasta", "./data/DRB1_aa_anno.txt")
print(aa_anno_df.shape)
aa_anno_df.head()


(440883, 4)


,id,allele1,allele2,hetero_vec
0,5723337,DRB1*04:01,DRB1*07:01,"0,0,0,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,..."
1,1089758,DRB1*01:03,DRB1*03:01,"0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,..."
2,3946752,DRB1*03:01,DRB1*07:01,"0,0,0,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,1,..."
3,5471666,DRB1*03:01,DRB1*16:01,"0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,..."
4,5133149,DRB1*01:01,DRB1*04:01,"0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,..."


In [9]:
import numpy as np
import pandas as pd
import random
from scipy.stats import entropy
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

def simulate_patient_data(aa_anno_df, n_samples=100):
    """
    从已有的0/1异质性向量中随机抽取样本，模拟病人状态
    返回 patient_id, hetero_vec, label
    """
    selected = aa_anno_df.sample(n=n_samples, random_state=42).reset_index(drop=True)
    
    # 模拟标签：一半为1（患病），一半为0（正常）
    labels = [1] * (n_samples // 2) + [0] * (n_samples - n_samples // 2)
    random.shuffle(labels)

    records = []
    for i, row in selected.iterrows():
        vec = list(map(int, row["hetero_vec"].split(",")))
        records.append({
            "patient_id": row["id"],
            "hetero_vec": vec,
            "label": labels[i]
        })
    return pd.DataFrame(records)



def compute_site_entropy(patients_df):
    """
    输入包含 hetero_vec 列的 DataFrame，输出每个位点的熵值
    """
    matrix = np.array(patients_df["hetero_vec"].tolist())  # shape: [N_samples, N_positions]
    entropies = []
    for i in range(matrix.shape[1]):
        counts = np.bincount(matrix[:, i], minlength=2)
        probs = counts / counts.sum()
        ent = entropy(probs, base=2)
        entropies.append(ent)
    return entropies

def logistic_regression_by_site(patients_df, entropies, entropy_thresh=0.0):
    """
    对所有多态位点进行逐位点逻辑回归分析
    """
    matrix = np.array(patients_df["hetero_vec"].tolist())
    labels = np.array(patients_df["label"])

    results = []
    for i, ent in enumerate(entropies):
        if ent <= entropy_thresh:
            continue  # 跳过无多态性位点
        x = matrix[:, i]
        X = sm.add_constant(x)
        model = sm.Logit(labels, X)
        try:
            res = model.fit(disp=False)
            pval = res.pvalues[1]
            coef = res.params[1]
            results.append({
                "Position": f"P{i+1}",
                "Entropy": ent,
                "Coefficient": coef,
                "P_value": pval
            })
        except Exception as e:
            continue  # 忽略无法拟合的模型

    df = pd.DataFrame(results)
    if not df.empty:
        _, adj_pvals, _, _ = multipletests(df["P_value"], method="fdr_bh")
        df["FDR"] = adj_pvals
    return df.sort_values("FDR")


In [10]:
# 读取已生成的 aa_anno_df（可通过 generate_aa_heterozygosity_file 返回）
patients_df = simulate_patient_data(aa_anno_df, n_samples=100)
patients_df.head()

,patient_id,hetero_vec,label
0,1690117,"[0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...",1
1,2501277,"[0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...",0
2,2257656,"[0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...",1
3,3373138,"[0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, ...",0
4,2163237,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0


In [20]:
entropies = compute_site_entropy(patients_df)
print(f"entropyists: {entropies[:10]}")
pbg_len = len(aa_anno_df["hetero_vec"][0].split(","))
print(f"peptide binding groove length: {pbg_len},  {len(entropies) == pbg_len}")

entropyists: [0.0, 0.0, 0.0, 0.9988455359952018, 0.9927744539878083, 0.5293608652873644, 0.9953784388202258, 0.4021791902022729, 0.8812908992306927, 0.0]
peptide binding groove length: 89,  True


In [21]:
regression_df = logistic_regression_by_site(patients_df, entropies)
print(regression_df.head())

   Position   Entropy  Coefficient   P_value       FDR
9       P23  1.000000     0.979096  0.017492  0.262379
23      P66  0.924819     1.105324  0.012777  0.262379
11      P26  0.855451     1.026799  0.028728  0.287279
17      P42  0.999711     0.728239  0.073383  0.550374
20      P55  0.985815    -0.575364  0.158906  0.608140


g:\miniconda\envs\hla_analysis\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


很好，我们先从\*\*stepwise logistic regression（逐步逻辑回归）\*\*的原理讲起，再在你提供的代码基础上添加 **双向搜索（forward + backward）** 的功能，使之等价于文献中提到的 `stepAIC` 实现。

---

## 🔍 一、Stepwise Logistic Regression 原理详解

### ✳️ 什么是逐步回归（Stepwise Regression）？

逐步回归是一种用于**特征选择**的回归策略。它通过**逐步添加或删除特征**，寻找能最小化某种评价指标（如 AIC）的最佳特征组合。

逐步回归有三种策略：

| 策略                   | 描述                             |
| -------------------- | ------------------------------ |
| Forward Selection    | 从空模型开始，每次引入一个最优特征，直到无法显著提高性能   |
| Backward Elimination | 从全模型开始，每次删除最无用的特征，直到删除任何特征都会变差 |
| Stepwise（双向）         | 同时尝试添加或删除特征，在每一步都评估添加或删除哪个更优   |

---

### ✳️ 什么是 AIC？

AIC（Akaike Information Criterion）是模型选择的标准，公式如下：

$$
\text{AIC} = 2k - 2\log L
$$

* $k$：模型的参数数（特征个数 + 截距）
* $L$：最大似然估计的值（模型对数据的拟合程度）

**目标是最小化 AIC**：兼顾了模型的**拟合能力（对数似然）**和**复杂度（特征数）**，防止过拟合。

---

### ✳️ 为什么 forward + backward 更合理？

* 仅 forward 会忽略后加入新特征后，已有特征可能变得不重要的情况；
* 仅 backward 可能初始模型太复杂（尤其变量很多时）；
* stepwise 在每一步同时尝试**添加和删除特征**，在模型结构上进行更全面搜索。

---

In [ ]:
import statsmodels.api as sm
import numpy as np

def stepwise_logistic_regression(patients_df, regression_df, fdr_thresh=0.3): # 注意修改FDR的阈值，一般是0.05
    """
    双向 stepwise logistic regression based on AIC.
    
    ✳️ stepwise 逻辑细节（forward + backward）：
    🔁 每轮迭代：
    Forward：尝试将未包含的特征加入模型，选择能最小化 AIC 的一个。

    Backward：尝试将已有的特征剔除模型，若剔除后 AIC 更小，则移除。

    若没有变量的添加或删除使 AIC 改善，则终止。

    """
    # 1. 提取候选特征
    selected = regression_df[regression_df["FDR"] <= fdr_thresh]["Position"].tolist()
    if not selected:
        raise ValueError("No features pass the FDR threshold.")
    feature_indices = [int(pos[1:]) - 1 for pos in selected]

    # 2. 构建数据
    matrix = np.array(patients_df["hetero_vec"].tolist())
    X_all = matrix[:, feature_indices]
    y = patients_df["label"].values

    included = []
    remaining = list(range(X_all.shape[1]))
    current_score, best_new_score = np.inf, np.inf

    while True:
        changed = False

        # 尝试添加每个候选变量
        candidates_to_add = [i for i in remaining if i not in included]
        add_scores = []
        for candidate in candidates_to_add:
            test_features = included + [candidate]
            X_step = sm.add_constant(X_all[:, test_features])
            try:
                model = sm.Logit(y, X_step).fit(disp=False)
                add_scores.append((model.aic, candidate, model))
            except Exception:
                continue
        if add_scores:
            add_scores.sort()
            best_add_aic, best_add_candidate, best_add_model = add_scores[0]
            if best_add_aic < current_score:
                included.append(best_add_candidate)
                current_score = best_add_aic
                best_model = best_add_model
                changed = True

        # 尝试删除已纳入的变量
        remove_scores = []
        for candidate in included:
            test_features = [f for f in included if f != candidate]
            if not test_features:
                continue
            X_step = sm.add_constant(X_all[:, test_features])
            try:
                model = sm.Logit(y, X_step).fit(disp=False)
                remove_scores.append((model.aic, candidate, model))
            except Exception:
                continue
        if remove_scores:
            remove_scores.sort()
            best_remove_aic, best_remove_candidate, best_remove_model = remove_scores[0]
            if best_remove_aic < current_score:
                included.remove(best_remove_candidate)
                current_score = best_remove_aic
                best_model = best_remove_model
                changed = True

        if not changed:
            break

    # 3. 返回最终结果
    final_features = [selected[i] for i in included]
    return final_features, best_model


In [29]:
features, final_model = stepwise_logistic_regression(patients_df, regression_df)
print("Selected Features:", features)
print(final_model.summary())


Selected Features: ['P66', 'P26', 'P23']
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  100
Model:                          Logit   Df Residuals:                       96
Method:                           MLE   Df Model:                            3
Date:                Sun, 20 Jul 2025   Pseudo R-squ.:                  0.1283
Time:                        17:14:06   Log-Likelihood:                -60.420
converged:                       True   LL-Null:                       -69.315
Covariance Type:            nonrobust   LLR p-value:                 0.0004862
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.6883      0.508     -3.322      0.001      -2.685      -0.692
x1             1.4782      0.504      2.931      0.003       0.490       2.467
x2         